# Neutral density

The original computes neutral density $\gamma^n$ with
`pygamma_n`, a wrapper around the Jackett & McDougall (1997)
algorithm. `pygamma_n` is not on conda-forge and is not an
ESMValTool dependency.

So this notebook uses it when it happens to be importable, and
otherwise falls back on a **patched potential density**: sigma
referenced to 0, 1000, 2000, 3000 and 4000 dbar, each patch offset
so the composite is continuous, after Reid (1994).

The fallback is an approximation and the plot says so. It follows
neutral surfaces well *within* each pressure patch and has a
discontinuous slope at the joins, so use it for water-mass
identification and not for quantitative neutral-surface work.

The original's own warning applies either way: $\gamma^n$ is
referenced to a 1980s climatology and should not be applied to
future projections.

---

Translated from the COSIMA recipe
[`Neutral_density.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Neutral_density.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import gsw
from esmvalcore.preprocessor import (climate_statistics,
                                     extract_region, regrid)

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', mip='Omon', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              dataset='ACCESS-CM2', timerange='1990/2009')

thetao = Dataset(short_name='thetao', **common)
so = thetao.copy(short_name='so')

def prepare(dataset):
    cube = climate_statistics(dataset.load(), operator='mean',
                              period='full')
    cube = regrid(cube, target_grid='1x1', scheme='linear')
    return extract_region(cube, start_longitude=0.,
                          end_longitude=360.,
                          start_latitude=-90., end_latitude=-30.)

thetao_cube, so_cube = prepare(thetao), prepare(so)

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


@contextlib.contextmanager
def quiet_gsw():
    """Silence gsw's masked-array warning for the enclosed call.

    ``gsw`` emits ``'where' used without 'out', expect uninitialized
    memory in output`` every time it is handed a masked array, which is
    every time here.  The uninitialised values sit only underneath the
    mask and never reach the result -- asserted in
    ``validation/test_cosima_common.py`` -- but the warning fires once
    per call and buries everything else a notebook prints.

    Scoped to this one message so genuine gsw warnings still surface.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=".*'where' used without 'out'.*",
            category=UserWarning)
        yield


def _require_gsw():
    if gsw is None:
        raise ImportError(
            "gsw (TEOS-10) is required for the density diagnostics; "
            "install it with `pixi add gsw`")


def pressure_from_depth(depth, lat):
    """Sea pressure in dbar from depth in metres (positive down)."""
    _require_gsw()
    depth = np.asarray(depth, dtype=float)
    lat = np.asarray(lat, dtype=float)
    return gsw.p_from_z(-depth, lat)


def absolute_salinity(practical_salinity, pressure, lon, lat):
    """Practical salinity (PSU) to absolute salinity (g/kg)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.SA_from_SP(practical_salinity, pressure, lon, lat)


def conservative_temperature(abs_salinity, potential_temperature):
    """Potential temperature (degC) to conservative temperature (degC)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.CT_from_pt(abs_salinity, potential_temperature)


def _broadcast_to_field(depth, lat, lon, shape):
    """Broadcast depth/lat/lon onto a ``(depth, lat, lon)`` field shape."""
    depth_b = np.broadcast_to(np.asarray(depth)[:, None, None], shape)
    lat_b = np.broadcast_to(np.asarray(lat)[None, :, None], shape)
    lon_b = np.broadcast_to(np.asarray(lon)[None, None, :], shape)
    return depth_b, lat_b, lon_b


def teos10_fields(thetao_cube, so_cube, temperature_is_conservative=False):
    """Absolute salinity, conservative temperature and pressure.

    Accepts fields with optional leading dimensions, such as time:
    ``(lat, lon)``, ``(depth, lat, lon)`` or
    ``(time, depth, lat, lon)``.  It returns masked arrays broadcast to
    the cube shape.  CMIP ``thetao`` is potential temperature, so it is
    converted unless the caller says otherwise (MOM5 ``temp`` is already
    conservative).
    """
    _require_gsw()
    salinity = masked_data(so_cube)
    temperature = masked_data(thetao_cube)

    if salinity.shape != temperature.shape:
        raise ValueError("thetao and so must have the same shape")

    def broadcast_1d(points, axis):
        """Broadcast a coordinate along its corresponding data axis."""
        reshape = [1] * salinity.ndim
        reshape[axis] = len(points)
        return np.broadcast_to(np.asarray(points).reshape(reshape),
                               salinity.shape)

    lat = lat_1d(so_cube)
    lon = lon_1d(so_cube)
    lat_axis = so_cube.coord_dims("latitude")[0]
    lon_axis = so_cube.coord_dims("longitude")[-1]
    depth_name = depth_coord_name(so_cube)
    if depth_name is None:
        depth_b = np.zeros(salinity.shape)
    else:
        depth = so_cube.coord(depth_name).points
        depth_axis = so_cube.coord_dims(depth_name)[0]
        depth_b = broadcast_1d(depth, depth_axis)
    lat_b = broadcast_1d(lat, lat_axis)
    lon_b = broadcast_1d(lon, lon_axis)
    pressure = pressure_from_depth(depth_b, lat_b)
    abs_sal = absolute_salinity(salinity, pressure, lon_b, lat_b)
    if temperature_is_conservative:
        cons_temp = temperature
    else:
        cons_temp = conservative_temperature(abs_sal, temperature)

    return abs_sal, cons_temp, pressure


def potential_density_anomaly(thetao_cube, so_cube, p_ref=2000,
                              temperature_is_conservative=False):
    """Potential density anomaly sigma_n in kg m-3.

    ``p_ref`` is the reference pressure in dbar; 0, 1000, 2000, 3000 and
    4000 map onto the ``gsw.sigma0``...``gsw.sigma4`` family.
    """
    _require_gsw()
    sigma_functions = {
        0: gsw.sigma0,
        1000: gsw.sigma1,
        2000: gsw.sigma2,
        3000: gsw.sigma3,
        4000: gsw.sigma4,
    }
    if p_ref not in sigma_functions:
        raise ValueError(
            f"p_ref must be one of {sorted(sigma_functions)}, got {p_ref}")
    abs_sal, cons_temp, _ = teos10_fields(
        thetao_cube, so_cube,
        temperature_is_conservative=temperature_is_conservative)
    with quiet_gsw():
        return np.ma.masked_invalid(
            sigma_functions[p_ref](abs_sal, cons_temp))


PATCH_LEVELS = (
    (0, 0.0, 500.0),
    (1000, 500.0, 1500.0),
    (2000, 1500.0, 2500.0),
    (3000, 2500.0, 3500.0),
    (4000, 3500.0, np.inf),
)


def patched_potential_density(thetao_cube, so_cube,
                              temperature_is_conservative=False):
    """Continuous patched potential density, an approximation to gamma_n.

    ``Neutral_density.ipynb`` uses ``pygamma_n`` (Jackett & McDougall
    1997), which is not installable from conda-forge.  This is the
    classical alternative: potential density referenced to 0, 1000,
    2000, 3000 and 4000 dbar, each patch offset so the composite is
    continuous across the joins, following Reid (1994).

    It is *not* neutral density.  It tracks neutral surfaces well within
    each patch and is discontinuous in slope at the joins; use it for
    water-mass identification, not for quantitative neutral-surface
    work.  When ``pygamma_n`` is importable the diagnostics use it
    instead.
    """
    _require_gsw()
    abs_sal, cons_temp, _ = teos10_fields(
        thetao_cube, so_cube,
        temperature_is_conservative=temperature_is_conservative)
    depth_name = depth_coord_name(so_cube)
    if depth_name is None:
        return potential_density_anomaly(
            thetao_cube, so_cube, p_ref=0,
            temperature_is_conservative=temperature_is_conservative)

    depth = np.asarray(so_cube.coord(depth_name).points, dtype=float)
    sigma_functions = {
        0: gsw.sigma0, 1000: gsw.sigma1, 2000: gsw.sigma2,
        3000: gsw.sigma3, 4000: gsw.sigma4,
    }

    result = np.ma.masked_all(abs_sal.shape)
    offset = 0.0
    previous = None
    with quiet_gsw():
        for p_ref, top, bottom in PATCH_LEVELS:
            levels = np.where((depth >= top) & (depth < bottom))[0]
            if levels.size == 0:
                continue
            sigma = sigma_functions[p_ref](abs_sal[levels],
                                           cons_temp[levels])
            if previous is not None:
                # Match the new patch to the old one at the shallowest
                # level of the new patch, so there is no jump.
                top_level = levels[0]
                overlap_previous = sigma_functions[previous](
                    abs_sal[top_level], cons_temp[top_level])
                overlap_new = sigma_functions[p_ref](
                    abs_sal[top_level], cons_temp[top_level])
                offset += float(
                    np.ma.median(overlap_previous - overlap_new + offset)
                    - offset)
            result[levels] = sigma + offset
            previous = p_ref
    return np.ma.masked_invalid(result)


def neutral_density(thetao_cube, so_cube, temperature_is_conservative=False):
    """Neutral density gamma_n, via ``pygamma_n`` when it is available.

    Falls back to :func:`patched_potential_density`.  The return value
    is ``(density, method)`` so callers can label their output honestly.
    """
    try:
        import pygamma_n
    except ImportError:
        return (
            patched_potential_density(
                thetao_cube, so_cube,
                temperature_is_conservative=temperature_is_conservative),
            "patched potential density (gamma_n approximation)",
        )

    _require_gsw()
    abs_sal, cons_temp, pressure = teos10_fields(
        thetao_cube, so_cube,
        temperature_is_conservative=temperature_is_conservative)
    with quiet_gsw():
        in_situ = gsw.t_from_CT(abs_sal, cons_temp, pressure)
    lat = lat_1d(so_cube)
    lon = lon_1d(so_cube)

    gamma = np.ma.masked_all(abs_sal.shape)
    for j in range(abs_sal.shape[1]):
        for i in range(abs_sal.shape[2]):
            column_s = abs_sal[:, j, i]
            column_t = in_situ[:, j, i]
            column_p = pressure[:, j, i]
            # pygamma_n's Fortran solver does not treat NaN as a missing
            # observation.  Passing a filled masked array therefore emits a
            # solver warning for partially wet columns and can contaminate
            # their result.  Give it only the physically observed levels;
            # the original vertical pressure values are retained.
            valid = (~np.ma.getmaskarray(column_s)
                     & ~np.ma.getmaskarray(column_t)
                     & ~np.ma.getmaskarray(column_p)
                     & np.isfinite(column_s)
                     & np.isfinite(column_t)
                     & np.isfinite(column_p))
            if np.count_nonzero(valid) < 2:
                continue
            values, _, _ = pygamma_n.gamma_n(
                np.asarray(column_s[valid]), np.asarray(column_t[valid]),
                np.asarray(column_p[valid]), lon[i], lat[j])
            gamma[valid, j, i] = values
    return np.ma.masked_invalid(gamma), "gamma_n (Jackett & McDougall 1997)"

## Compute

In [ ]:
gamma, method = neutral_density(thetao_cube, so_cube)
print('method:', method)

section = np.ma.mean(gamma, axis=-1)   # zonal mean
depth = thetao_cube.coord(depth_coord_name(thetao_cube)).points
lat = lat_1d(thetao_cube)

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

mesh = ax.pcolormesh(lat, depth, section, cmap='viridis',
                     shading='auto')
contours = ax.contour(lat, depth, section, colors='k',
                      linewidths=0.6, levels=14)
ax.clabel(contours, inline=True, fontsize=7, fmt='%.2f')

ax.invert_yaxis()
ax.set_xlabel('Latitude')
ax.set_ylabel('Depth (m)')
ax.set_title(method + '\nzonal mean')
fig.colorbar(mesh, ax=ax, shrink=0.85, label='kg m$^{-3}$')
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()